# Getting started — the published graphs, no clone

For a developer who wants to query cdisc-for-all. Nothing to clone and nothing to build: the notebook reads `catalog.ttl` from the GitHub Pages site, downloads the graphs it lists (checked against the SHA-256 in the catalogue), loads them into Oxigraph, and follows Glucose (SDTM CT code C105585) one hop at a time. Each hop names the link predicate it follows.

Needs Python 3 with `pyoxigraph` and `pandas`:

    python3 -m venv ~/venvs/cdisc-for-all
    ~/venvs/cdisc-for-all/bin/pip install pyoxigraph==0.5.13 pandas==3.1.1 jupyter

The graphs (about 115 MB) are kept in `~/cdisc-for-all-data/` and downloaded again only when the catalogue lists a different checksum.

Reading the IRIs: `def/<source>/<table>#<column>` is a published column (the literal is the cell as published), `id/<source>/<table>/<key>` is a row, `link/...` is a link on a published key, `rule/...` a link by a stated rule. NCIt concepts keep NCI's own IRIs. The links are listed with their rules in `llms.txt` and in `schema/links.yaml`.

In [ ]:
import hashlib
import urllib.request
from pathlib import Path

import pandas as pd
from pyoxigraph import NamedNode, RdfFormat, Store

SITE = 'https://kerfors.github.io/cdisc-for-all/'
DATA = Path.home() / 'cdisc-for-all-data'
DATA.mkdir(exist_ok=True)


def fetch(url):
    with urllib.request.urlopen(url) as r:
        return r.read()


def table(store, query):
    """A SELECT result as a DataFrame of plain values (IRIs and literals as text). The query reads
    the union of the named graphs as its default graph."""
    result = store.query(query, use_default_graph_as_union=True)
    names = [v.value for v in result.variables]
    return pd.DataFrame([[None if s[n] is None else s[n].value for n in names] for s in result], columns=names)

## 1. The catalogue

`catalog.ttl` is DCAT + VoID: each graph is a `dcat:Dataset` with its version, licence or rights and `void:triples`; each file is a `dcat:Distribution` with its download URL and SHA-256. The sign-in sources (SDTMIG, SDTM Model) are listed without files.

In [ ]:
catalog = Store()
catalog.load(fetch(SITE + 'catalog.ttl'), format=RdfFormat.TURTLE, base_iri=SITE)
graphs = table(catalog, '''
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX void: <http://rdfs.org/ns/void#>
PREFIX spdx: <http://spdx.org/rdf/terms#>
SELECT ?dataset ?title ?version ?triples ?url ?sha256 WHERE {
    ?dataset a void:Dataset ; dcterms:title ?title .
    OPTIONAL { ?dataset dcat:version ?version }
    OPTIONAL { ?dataset void:triples ?triples ;
                        dcat:distribution ?d .
               ?d dcat:mediaType <https://www.iana.org/assignments/media-types/text/turtle> ;
                  dcat:downloadURL ?url ;
                  spdx:checksum/spdx:checksumValue ?sha256 }
} ORDER BY ?dataset''')
graphs[['title', 'version', 'triples', 'url']]

## 2. Download and load

Every graph with a Turtle file goes into one store, each in its own named graph (the dataset IRI), so a query can say where a triple comes from. A file whose SHA-256 is not the catalogue's stops the notebook.

In [ ]:
store = Store()
for row in graphs.dropna(subset=['url']).itertuples():
    path = DATA / row.url[len(SITE):].replace('/', '__')
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != row.sha256:
        path.write_bytes(fetch(row.url))
    if hashlib.sha256(path.read_bytes()).hexdigest() != row.sha256:
        raise ValueError(f'{row.url}: SHA-256 is not the one catalog.ttl lists')
    store.bulk_load(path=str(path), format=RdfFormat.TURTLE, to_graph=NamedNode(row.dataset))
    print(f'{row.url[len(SITE):]:45} {int(row.triples):>9} triples')
print(len(store), 'quads in the store')

## 3. Hop 0 — the SDTM CT terms with code C105585

Base graph `sdtm-ct`, columns as published. The same code is a term in Laboratory Test Code (GLUC) and in Laboratory Test Name (Glucose). Change `CODE` for another test.

In [ ]:
CODE = 'C105585'
P = '''
PREFIX ct: <https://kerfors.github.io/cdisc-for-all/def/sdtm-ct/SDTM%20Terminology#>
PREFIX ctl: <https://kerfors.github.io/cdisc-for-all/link/sdtm-ct/SDTM%20Terminology#>
PREFIX ncit: <https://kerfors.github.io/cdisc-for-all/def/ncit-flat/Thesaurus#>
PREFIX ncitl: <https://kerfors.github.io/cdisc-for-all/link/ncit-flat/Thesaurus#>
PREFIX bc: <https://kerfors.github.io/cdisc-for-all/def/cosmos/cdisc_biomedical_concepts_latest#>
PREFIX bcl: <https://kerfors.github.io/cdisc-for-all/link/cosmos/cdisc_biomedical_concepts_latest#>
PREFIX dss: <https://kerfors.github.io/cdisc-for-all/def/cosmos/cdisc_sdtm_dataset_specializations_latest#>
PREFIX dssl: <https://kerfors.github.io/cdisc-for-all/link/cosmos/cdisc_sdtm_dataset_specializations_latest#>
'''
table(store, P + f'''
SELECT ?term ?codelist_name ?submission_value ?definition WHERE {{
    ?term ct:Code "{CODE}" ;
          ct:Codelist%20Name ?codelist_name ;
          ct:CDISC%20Submission%20Value ?submission_value ;
          ct:CDISC%20Definition ?definition .
}}''')

## 4. Hop 1 — to the NCIt concept

Link `link/sdtm-ct/SDTM Terminology#Code` (published key: the Code column names an NCIt concept). Base graph `ncit-flat`, with the concept's ancestors by `link/ncit-flat/Thesaurus#parents`.

In [ ]:
table(store, P + f'''
SELECT DISTINCT ?concept ?display_name ?semantic_type ?ancestor ?ancestor_name WHERE {{
    ?term ct:Code "{CODE}" ; ctl:Code ?concept .
    ?concept ncit:display%20name ?display_name ;
             ncit:semantic%20type ?semantic_type ;
             ncitl:parents+ ?ancestor .
    ?ancestor ncit:display%20name ?ancestor_name .
}}''')

## 5. Hop 2 — to the COSMoS Biomedical Concept

No link of ours joins a CT term to a BC: the term's NCIt concept is the `ncit_code` of the BC rows (link `link/cosmos/cdisc_biomedical_concepts_latest#ncit_code`). One BC row per Data Element Concept; a BC without DECs has one row with an empty `dec_id`.

In [ ]:
table(store, P + f'''
SELECT DISTINCT ?bc_id ?short_name ?dec_id ?dec_label ?data_type WHERE {{
    ?term ct:Code "{CODE}" ; ctl:Code ?concept .
    ?bc_row bcl:ncit_code ?concept ;
            bc:bc_id ?bc_id ; bc:short_name ?short_name .
    OPTIONAL {{ ?bc_row bc:dec_id ?dec_id }}
    OPTIONAL {{ ?bc_row bc:dec_label ?dec_label }}
    OPTIONAL {{ ?bc_row bc:data_type ?data_type }}
}} ORDER BY ?dec_id''')

## 6. Hop 3 — to the Dataset Specializations

Each Dataset Specialization row that names one of those DECs, by link `link/cosmos/cdisc_sdtm_dataset_specializations_latest#dec_id` (published key: bc_id + dec_id). Grouped by Specialization (`vlm_group_id`).

In [ ]:
table(store, P + f'''
SELECT ?vlm_group_id ?short_name (GROUP_CONCAT(DISTINCT ?variable; separator=", ") AS ?variables_with_a_dec) WHERE {{
    ?term ct:Code "{CODE}" ; ctl:Code ?concept .
    ?bc_row bcl:ncit_code ?concept .
    ?row dssl:dec_id ?bc_row ;
         dss:vlm_group_id ?vlm_group_id ; dss:short_name ?short_name ; dss:sdtm_variable ?variable .
}} GROUP BY ?vlm_group_id ?short_name ORDER BY ?vlm_group_id''')

## 7. Hop 4 — to the CT terms the Specializations bind

From those rows to SDTM CT term rows: by link `link/...#assigned_term` (published key: assigned term within its codelist) and by the rule link `rule/...#value_list/sdtm-ct/SDTM Terminology` (each value of value_list, split at ";", is a Submission Value of the codelist). The `how` column says which.

In [ ]:
table(store, P + f'''
SELECT ?dec_label ?variable ?how ?codelist_name ?submission_value (COUNT(DISTINCT ?row) AS ?rows) WHERE {{
    ?term ct:Code "{CODE}" ; ctl:Code ?concept .
    ?bc_row bcl:ncit_code ?concept ; bc:dec_label ?dec_label .
    ?row dssl:dec_id ?bc_row ; dss:sdtm_variable ?variable .
    {{ ?row dssl:assigned_term ?bound . BIND("assigned_term (key)" AS ?how) }}
    UNION
    {{ ?row <https://kerfors.github.io/cdisc-for-all/rule/cosmos/cdisc_sdtm_dataset_specializations_latest#value_list/sdtm-ct/SDTM%20Terminology> ?bound . BIND("value_list (rule)" AS ?how) }}
    ?bound ct:Codelist%20Name ?codelist_name ; ct:CDISC%20Submission%20Value ?submission_value .
}} GROUP BY ?dec_label ?variable ?how ?codelist_name ?submission_value
ORDER BY ?dec_label ?variable ?submission_value''')

## 8. Where a triple comes from

Each graph is in its own named graph, so the hops can be traced back to their source. Here: which graph states each link used above.

In [ ]:
table(store, P + f'''
SELECT ?graph ?link (COUNT(*) AS ?triples) WHERE {{
    VALUES ?link {{ ctl:Code bcl:ncit_code dssl:dec_id dssl:assigned_term <https://kerfors.github.io/cdisc-for-all/rule/cosmos/cdisc_sdtm_dataset_specializations_latest#value_list/sdtm-ct/SDTM%20Terminology> }}
    GRAPH ?graph {{ ?s ?link ?o }}
}} GROUP BY ?graph ?link ORDER BY ?link''')

## 9. Next: the whole test concept graph in one query

`examples/test_concept_graph.rq` does all of this in one CONSTRUCT and adds the BC parents, the DECs' NCIt concepts, the code–CUI map rows and the other tests of the BC. Pages serves it next to the graphs. The query reads the default graph, so it runs over the union of the named graphs (`use_default_graph_as_union`). `examples/test_concept.py` in the repository runs it per C-code and writes a page.

In [ ]:
import re

query = fetch(SITE + 'examples/test_concept_graph.rq').decode('utf-8')
query = re.sub(r'VALUES \?code \{ "C\d+" \}', f'VALUES ?code {{ "{CODE}" }}', query)
result = list(store.query(query, use_default_graph_as_union=True))
print(len(result), 'triples in the test concept graph of', CODE)